# Linear, ReLU, Sigmoid, and Softmax Examples

This notebook shows simple PyTorch tensor implementations of:

- Linear: `y = Wx + b`
- ReLU: `max(0, x)`
- Sigmoid: `1 / (1 + exp(-x))`
- Softmax: `exp(x) / sum(exp(x))`

In [4]:
import torch

torch.set_printoptions(precision=4, sci_mode=False)

## 1) Linear function

For vectors/matrices, a linear layer is often written as:

\[ y = xW + b \]

where `x` is a batch of inputs, `W` is weights, and `b` is bias.

With `torch.nn.Linear`, `W` and `b` are stored inside the layer object, so you call `layer(x)`.

In [5]:

x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0]
])


layer = torch.nn.Linear(in_features=3, out_features=1, bias=True)

y_linear = layer(x)
print('x input:\n', x)
print('Linear output:\n', y_linear)

x input:
 tensor([[1., 2., 3.],
        [4., 5., 6.]])
Linear output:
 tensor([[-1.4613],
        [-2.5587]], grad_fn=<AddmmBackward0>)


In [7]:
target = torch.tensor([
    [2.5],
    [0.7]
])

criterion = torch.nn.MSELoss()
loss = criterion(y_linear, target)

print('y_linear:\n', y_linear)
print('Target:\n', target)
print('MSE loss:', loss.item())

y_linear:
 tensor([[-1.4613],
        [-2.5587]], grad_fn=<AddmmBackward0>)
Target:
 tensor([[2.5000],
        [0.7000]])
MSE loss: 13.155519485473633


## 2) ReLU activation

ReLU keeps positive values and clips negatives to 0.

In [8]:
x_vals = torch.tensor([-3.0, -0.5, 0.0, 0.7, 2.3])
y_relu = torch.relu(x_vals)

print('Input :', x_vals)
print('ReLU  :', y_relu)

Input : tensor([-3.0000, -0.5000,  0.0000,  0.7000,  2.3000])
ReLU  : tensor([0.0000, 0.0000, 0.0000, 0.7000, 2.3000])


## 3) Sigmoid activation

Sigmoid maps values to the range `(0, 1)`.

In [9]:
x_vals = torch.tensor([-6.0, -2.0, 0.0, 2.0, 6.0])
y_sigmoid = torch.sigmoid(x_vals)

print('Input   :', x_vals)
print('Sigmoid :', y_sigmoid)

Input   : tensor([-6., -2.,  0.,  2.,  6.])
Sigmoid : tensor([0.0025, 0.1192, 0.5000, 0.8808, 0.9975])


In [10]:
# Toy loss example: compare sigmoid predictions to binary targets
target_probs = torch.tensor([0.0, 0.0, 1.0, 1.0, 1.0])
criterion = torch.nn.BCELoss()
loss = criterion(y_sigmoid, target_probs)

print('Predicted (y_sigmoid):', y_sigmoid)
print('Target labels (0/1) :', target_probs)
print('BCE loss             :', loss.item())

Predicted (y_sigmoid): tensor([0.0025, 0.1192, 0.5000, 0.8808, 0.9975])
Target labels (0/1) : tensor([0., 0., 1., 1., 1.])
BCE loss             : 0.1903909146785736


In [11]:
# Toy loss example: BCE with logits (use raw logits, no sigmoid first)
logits = x_vals
criterion_logits = torch.nn.BCEWithLogitsLoss()
loss_logits = criterion_logits(logits, target_probs)

print('Raw logits          :', logits)
print('Sigmoid(logits)     :', torch.sigmoid(logits))
print('Target labels (0/1) :', target_probs)
print('BCEWithLogits loss  :', loss_logits.item())

Raw logits          : tensor([-6., -2.,  0.,  2.,  6.])
Sigmoid(logits)     : tensor([0.0025, 0.1192, 0.5000, 0.8808, 0.9975])
Target labels (0/1) : tensor([0., 0., 1., 1., 1.])
BCEWithLogits loss  : 0.190390944480896


## 4) Softmax activation

Softmax converts a vector of scores into probabilities that sum to 1.

In [12]:
scores = torch.tensor([2.0, 1.0, 0.1])
probs = torch.softmax(scores, dim=0)

print('Scores   :', scores)
print('Softmax  :', probs)
print('Sum probs:', probs.sum())

Scores   : tensor([2.0000, 1.0000, 0.1000])
Softmax  : tensor([0.6590, 0.2424, 0.0986])
Sum probs: tensor(1.0000)


In [13]:
# Toy loss example: CrossEntropyLoss expects raw logits (softmax is fused internally)
logits_ce = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.5, 2.2, -1.0]
])
target_class = torch.tensor([0, 1])

ce = torch.nn.CrossEntropyLoss()
loss_ce = ce(logits_ce, target_class)

# Equivalent manual form: log_softmax + NLLLoss
loss_manual = torch.nn.NLLLoss()(torch.log_softmax(logits_ce, dim=1), target_class)

print('Raw logits:\n', logits_ce)
print('Softmax probs:\n', torch.softmax(logits_ce, dim=1))
print('Target class indices:', target_class)
print('CrossEntropy loss   :', loss_ce.item())
print('Manual (logsm+NLL) :', loss_manual.item())

Raw logits:
 tensor([[ 2.0000,  1.0000,  0.1000],
        [ 0.5000,  2.2000, -1.0000]])
Softmax probs:
 tensor([[0.6590, 0.2424, 0.0986],
        [0.1493, 0.8174, 0.0333]])
Target class indices: tensor([0, 1])
CrossEntropy loss   : 0.30935055017471313
Manual (logsm+NLL) : 0.30935055017471313
